# Import Modules and Create Spark Session

In [1]:
import json
import gzip
import pandas as pd
import numpy as np
import seaborn as sns
from urllib.request import urlopen
from pyspark.sql.functions import mean, col, udf, split, explode, array_remove, avg
from pyspark.sql.functions import concat_ws, size, count, countDistinct, log10, rand, rank, first
from pyspark.sql.functions import split, collect_set, collect_list, flatten
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, IndexToString, Word2Vec, VectorAssembler
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.stat import Summarizer
from pyspark.mllib.evaluation import RegressionMetrics
from sklearn.metrics.pairwise import cosine_similarity
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pyspark.sql.window import Window
import string
import nltk
nltk.download('all')
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')
nltk.download('punkt')
from datetime import datetime

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package basque_grammars to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package basque_grammars is already up-to-date!
[nltk_data]    | Downloading package bcp47 to /root/nltk_data...
[nltk_data]    |   Package bcp47 is already up-to-dat

In [2]:
spark = SparkSession \
    .builder \
    .appName('models') \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.kryoserializer.buffer.max", "10g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "30g") \
    .config("fraction spark.storage.memoryFraction", "0.8") \
    .config("spark.executor.memory", "20g") \
    .getOrCreate()

# Loading Dataset 

In [3]:
!wget --no-check-certificate https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Luxury_Beauty.json.gz

--2023-04-27 13:54:59--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Luxury_Beauty.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 73193215 (70M) [application/x-gzip]
Saving to: ‘Luxury_Beauty.json.gz’

Luxury_Beauty.json. 100%[===================>]  69.80M  19.0MB/s    in 4.7s    

2023-04-27 13:55:04 (14.8 MB/s) - ‘Luxury_Beauty.json.gz’ saved [73193215/73193215]



## With Hadoop (does not work on Sentiment)

In [5]:
# !hadoop fs -put Magazine_Subscriptions.json.gz /

In [6]:
# %%time
# data = spark.read.json('hdfs://st446-cluster-2-m:8051/Magazine_Subscriptions.json.gz')
# data.show(5)

## With Pandas (Works on all models)

In [4]:
%%time
### load the meta data
data = []
with gzip.open('Luxury_Beauty.json.gz') as f:
    for l in f:
        data.append(json.loads(l.strip()))
        
# convert list into pandas dataframe
df = pd.DataFrame.from_dict(data) 

# only keep columns we want
df = df[['overall', 'reviewerID', 'asin', 'reviewText']]

# Get subset of dataset
# df = df.sort_values(by="reviewerID")
# df = df[:1000]

# Define Schema
from pyspark.sql.types import *
schema = StructType([
    StructField("overall", DoubleType(), True),
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("reviewText", StringType(), True),
])

reviews = spark.createDataFrame(df, schema=schema) 

CPU times: user 13.7 s, sys: 1.05 s, total: 14.8 s
Wall time: 17.2 s


# Changing column names and split into train test

In [5]:
%%time
reviews = reviews.select(col("reviewerID"), col("asin"), col("reviewText"), col("overall"))\
                .withColumnRenamed("asin", "productID")\
                .withColumnRenamed("overall", "rating")

reviews_train, reviews_test = reviews.randomSplit([0.8, 0.2], seed=42)
reviews_test = reviews_test.drop("reviewText")
print("reviews_train shape: ", reviews_train.count())
print("reviews_test shape: ", reviews_test.count())

23/04/27 13:55:31 WARN org.apache.spark.scheduler.TaskSetManager: Stage 0 contains a task of very large size (36243 KiB). The maximum recommended task size is 1000 KiB.
23/04/27 13:55:37 WARN org.apache.spark.scheduler.TaskSetManager: Stage 3 contains a task of very large size (36243 KiB). The maximum recommended task size is 1000 KiB.


reviews_train shape:  459706


reviews_test shape:  114922
CPU times: user 25.2 ms, sys: 0 ns, total: 25.2 ms
Wall time: 8.31 s


# Models

## 1. Baseline Model

In [6]:
%%time

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for i in partitions:
    
    start_time = datetime.now()
    
    reviews_train = reviews_train.repartition(i).cache()
    reviews_test = reviews_test.repartition(i).cache()

    alpha = reviews_train.select(mean("rating")).collect()[0][0]

    reviewer_means = reviews_train.groupBy("reviewerID")\
                                  .agg(avg("rating").alias("reviewer_mean"))

    product_means = reviews_train.groupBy("productID")\
                                 .agg(avg("rating").alias("product_mean"))

    reviews_test_with_means = reviews_test.join(reviewer_means, "reviewerID", "left")\
                                            .join(product_means, "productID", "left")\
                                            .fillna(alpha, subset=["reviewer_mean", "product_mean"])

    def baseline_model(row):
        reviewer_mean = row["reviewer_mean"]
        product_mean = row["product_mean"]
        baseline_rating = alpha + (reviewer_mean - alpha) + (product_mean - alpha)
        return (row["productID"], row["reviewerID"], row["rating"], baseline_rating)

    reviews_test_with_preds = reviews_test_with_means.rdd.map(baseline_model)
    reviews_test_with_preds = reviews_test_with_preds.toDF(["productID", "reviewerID", "rating", "baseline_pred"])

    # Getting Metrics
    baseline_val_and_preds = reviews_test_with_preds.select(["rating", "baseline_pred"])
    baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
    metrics = RegressionMetrics(baseline_val_and_preds)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([metrics.meanSquaredError, metrics.rootMeanSquaredError, metrics.meanAbsoluteError, total_time])
    
    print(f'Partition {i} Done. Start Time = {start_time}, End Time = {end_time}.')

23/04/27 13:56:10 WARN org.apache.spark.scheduler.TaskSetManager: Stage 6 contains a task of very large size (36243 KiB). The maximum recommended task size is 1000 KiB.
23/04/27 13:56:17 WARN org.apache.spark.scheduler.TaskSetManager: Stage 19 contains a task of very large size (36243 KiB). The maximum recommended task size is 1000 KiB.


Partition 2 Done. Start Time = 2023-04-27 13:56:09.582875, End Time = 2023-04-27 13:56:20.158021.


Partition 4 Done. Start Time = 2023-04-27 13:56:21.376505, End Time = 2023-04-27 13:56:25.237919.


Partition 8 Done. Start Time = 2023-04-27 13:56:26.052604, End Time = 2023-04-27 13:56:28.976462.


Partition 16 Done. Start Time = 2023-04-27 13:56:29.939465, End Time = 2023-04-27 13:56:33.068404.


Partition 64 Done. Start Time = 2023-04-27 13:56:33.984245, End Time = 2023-04-27 13:56:38.413408.


Partition 128 Done. Start Time = 2023-04-27 13:56:40.609178, End Time = 2023-04-27 13:56:48.999859.


Partition 512 Done. Start Time = 2023-04-27 13:56:51.509891, End Time = 2023-04-27 13:57:07.063847.
CPU times: user 547 ms, sys: 160 ms, total: 707 ms
Wall time: 1min 3s


In [7]:
baseline_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
baseline_partitions

,MSE,RMSE,MAE,Time (s)
2,1.489986,1.22065,0.861779,10.575146
4,1.489986,1.22065,0.861779,3.861414
8,1.489986,1.22065,0.861779,2.923858
16,1.489986,1.22065,0.861779,3.128939
64,1.489986,1.22065,0.861779,4.429163
128,1.489986,1.22065,0.861779,8.390681
512,1.489986,1.22065,0.861779,15.553956


## 2. ALS

In [8]:
%%time
# get train and test
stringIndexer = StringIndexer(inputCol='reviewerID', outputCol='reviewer')
model_train_reviewer = stringIndexer.fit(reviews_train)
train = model_train_reviewer.transform(reviews_train)
model_test_reviewer = stringIndexer.fit(reviews_test)
test = model_test_reviewer.transform(reviews_test)

stringIndexer = StringIndexer(inputCol='productID', outputCol='product')
model_train_product = stringIndexer.fit(train)
train = model_train_product.transform(train)
model_test_product = stringIndexer.fit(test)
test = model_test_product.transform(test)

train = train.drop("reviewText").select('reviewer', 'product', 'rating')
test = test.drop("reviewText").select('reviewer', 'product', 'rating')

train.cache()
test.cache()

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for i in partitions:
    
    start_time = datetime.now()
    
    train = train.repartition(i).cache()
    test = test.repartition(i).cache()

    als = ALS(
        rank=10,
        maxIter=15,
        regParam=0.2,
        userCol='reviewer',
        itemCol='product',
        ratingCol='rating',  
        coldStartStrategy='drop',
        nonnegative=True
    )

    # fit the model
    model = als.fit(train)

    # performing predictions using the testing data
    prediction = model.transform(test)
    # prediction.cache().show()

    # evaluate the model by computing the RMSE on the test data
    mse_evaluator_als = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
    rmse_evaluator_als = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
    mae_evaluator_als = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

    mse = mse_evaluator_als.evaluate(prediction)
    rmse = rmse_evaluator_als.evaluate(prediction)
    mae = mae_evaluator_als.evaluate(prediction)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([mse, rmse, mae, total_time])
    
    print(f'Partition {i} Done. Start Time = {start_time}, End Time = {end_time}.')

23/04/27 13:57:39 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 12.0 MiB
23/04/27 13:58:01 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 13:58:02 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:03 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:03 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:05 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:06 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:07 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:08 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary 

Partition 2 Done. Start Time = 2023-04-27 13:57:37.324415, End Time = 2023-04-27 13:58:47.370359.


23/04/27 13:58:48 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 13:58:48 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 13:58:49 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:50 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:50 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:51 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:51 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:53 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:58:53 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary w

Partition 4 Done. Start Time = 2023-04-27 13:58:47.370471, End Time = 2023-04-27 13:59:23.879712.


23/04/27 13:59:24 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 13:59:25 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 13:59:25 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:26 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:26 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:27 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:28 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:29 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 13:59:29 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary w

Partition 8 Done. Start Time = 2023-04-27 13:59:23.879855, End Time = 2023-04-27 14:00:00.296824.


... (4 outputs left out)

23/04/27 14:01:21 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 14:01:23 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:23 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:26 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:28 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:29 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:31 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:32 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:01:32 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary w

Partition 128 Done. Start Time = 2023-04-27 14:01:20.227509, End Time = 2023-04-27 14:02:14.919059.


23/04/27 14:02:15 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.7 MiB
23/04/27 14:02:18 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:19 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:30 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:36 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:37 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:42 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:43 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary with size 5.8 MiB
23/04/27 14:02:44 WARN org.apache.spark.scheduler.DAGScheduler: Broadcasting large task binary w

Partition 512 Done. Start Time = 2023-04-27 14:02:14.919170, End Time = 2023-04-27 14:04:05.816805.
CPU times: user 1.64 s, sys: 307 ms, total: 1.94 s
Wall time: 6min 40s


In [9]:
als_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
als_partitions

,MSE,RMSE,MAE,Time (s)
2,3.669933,1.915707,1.605459,70.045944
4,3.669933,1.915707,1.605459,36.509241
8,3.669933,1.915707,1.605459,36.416969
16,3.669933,1.915707,1.605459,36.241148
64,3.669933,1.915707,1.605459,43.689180
128,3.669933,1.915707,1.605459,54.691550
512,3.669933,1.915707,1.605459,110.897635


## 3. Similarity Based Model

In [10]:
%%time

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for j in partitions:
    
    start_time = datetime.now()
    
    reviews_train = reviews_train.repartition(i).cache()
    reviews_test = reviews_test.repartition(i).cache()
    
    # Data Preprocessing
    reviews_train_model = reviews_train.withColumn("words", split("reviewText", "\\.|!|\\?")) \
                                        .withColumn("words", explode("words")) \
                                        .withColumn("words", split("words", "\\s+")) \
                                        .withColumn("words", array_remove("words", ""))\
                                        .drop("reviewText")

    # Training Model (Set # of Partitions)
    word2Vec = Word2Vec(vectorSize=100, seed=42, numPartitions=16, inputCol="words", outputCol="model")
    sb_model = word2Vec.fit(reviews_train_model)

    # Fitting Data on Model
    vectors = sb_model.transform(reviews_train_model)
    vectors = vectors.drop("words")

    # Getting average word2vec vector scores for each rating (1 to 5) for relevant reviews & products (10 scores in total)
    # Add each score for each (reviewerID, productID) pair as an additional column - 10 new columns total
    assembler = VectorAssembler(inputCols=["model"], outputCol="features")
    vectors_assembled = assembler.transform(vectors)

    vectors_reviewers = vectors_assembled.groupBy('reviewerID', 'rating')\
                                        .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                        .withColumnRenamed("rating", "rating_")

    vectors_products = vectors_assembled.groupBy('productID', 'rating')\
                                        .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                        .withColumnRenamed("rating", "rating_")

    vectors_all = reviews_test
    for i in range(1,6):
        vectors_all = vectors_all.join(vectors_reviewers.filter(f"rating = {i}"), on='reviewerID', how='left')\
                                .withColumnRenamed('avg_vector', f"r_{i}")\
                                .drop("rating_")

        vectors_all = vectors_all.join(vectors_products.filter(f"rating = {i}"), on='productID', how='left')\
                                .withColumnRenamed('avg_vector', f"p_{i}")\
                                .drop("rating_")

    # Calculate cosine similarity scores and return predicted rating
    def cosine_similarity_map(row):
        cosine_scores = []

        for i in range(1,6):
            reviewer_vector = row[f"r_{i}"] #switch to index? row[3], row[5], ...
            product_vector = row[f"p_{i}"] #switch to index? row[3], row[5], ...

            if (reviewer_vector is None) or (product_vector is None):
                cosine_scores.append(None)

            else:
                cosine_score = cosine_similarity(reviewer_vector.reshape(1,100), product_vector.reshape(1,100))
                cosine_scores.append(cosine_score)


        try:
            predicted_rating = float(np.nanargmax(np.array(cosine_scores, dtype=np.float64))+1)
        except ValueError:
            predicted_rating = round(alpha,0)

        return (row[0], row[1], row["rating"], predicted_rating)


    vectors_all_pred = vectors_all.rdd.map(cosine_similarity_map)
    reviews_test_with_preds = vectors_all_pred.toDF(["productID", "reviewerID", "rating", "sb_pred"])

    # Getting Metrics
    baseline_val_and_preds = reviews_test_with_preds.select(["rating", "sb_pred"])
    baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
    metrics = RegressionMetrics(baseline_val_and_preds)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([metrics.meanSquaredError, metrics.rootMeanSquaredError, metrics.meanAbsoluteError, total_time])
    
    print(f'Partition {j} Done. Start Time = {start_time}, End Time = {end_time}.')

23/04/27 14:05:58 WARN com.github.fommil.netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
23/04/27 14:05:58 WARN com.github.fommil.netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


Partition 2 Done. Start Time = 2023-04-27 14:04:41.365797, End Time = 2023-04-27 14:07:22.153528.


Partition 4 Done. Start Time = 2023-04-27 14:07:39.436434, End Time = 2023-04-27 14:09:45.928462.


Partition 8 Done. Start Time = 2023-04-27 14:09:57.423577, End Time = 2023-04-27 14:12:04.255700.


Partition 16 Done. Start Time = 2023-04-27 14:12:16.357417, End Time = 2023-04-27 14:14:27.734195.


Partition 64 Done. Start Time = 2023-04-27 14:14:39.971794, End Time = 2023-04-27 14:16:48.634032.


Partition 128 Done. Start Time = 2023-04-27 14:17:01.368735, End Time = 2023-04-27 14:19:10.830195.


Partition 512 Done. Start Time = 2023-04-27 14:19:23.449351, End Time = 2023-04-27 14:21:33.832030.
CPU times: user 1.67 s, sys: 405 ms, total: 2.07 s
Wall time: 17min 5s


In [11]:
similarity_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
similarity_partitions

,MSE,RMSE,MAE,Time (s)
2,1.587050,1.259782,0.886175,160.787731
4,1.584188,1.258645,0.885000,126.492028
8,1.583100,1.258213,0.884678,126.832123
16,1.585989,1.259360,0.885862,131.376778
64,1.586624,1.259613,0.885783,128.662238
128,1.585519,1.259174,0.885427,129.461460
512,1.584544,1.258787,0.885113,130.382679
